In [1]:
import geopandas as gpd
from shapely.geometry import Point
import folium
import pandas as pd
from shapely import wkt
import json

## ***load grid shapefile***

In [20]:
# Load your grid shapefile or GeoDataFrame (in EPSG:4326)
grid = gpd.read_file("C:/Users/Giorgos/Desktop/HMMY/10ο Εξάμηνο/Διπλωματική/2. Datasets/ELSTAT/ΔΗΜΟΤΙΚΕΣ ΚΟΙΝΟΤΗΤΕΣ - shapefiles/grid/magnesia-grid.shp").to_crs("EPSG:4326")
grid['centroid_geom'] = grid.geometry.centroid
print(grid.columns)

Index(['id', 'left', 'top', 'right', 'bottom', 'row_index', 'col_index',
       'centroi_lo', 'centroi_la', 'geometry', 'centroid_geom'],
      dtype='object')


C:\Users\Giorgos\AppData\Local\Temp\ipykernel_23620\80253860.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid['centroid_geom'] = grid.geometry.centroid


In [21]:
# For the grid shapefile
print("Grid geometry type:", grid.geometry.geom_type.unique())

# Inspect first 5 polygons in the grid
print("Grid geometries:")
for geom in grid.geometry.head():
    if geom.geom_type == "Polygon":
        print(f"Points: {len(geom.exterior.coords)}")

Grid geometry type: ['Polygon']
Grid geometries:
Points: 5
Points: 5
Points: 5
Points: 5
Points: 5


## ***load municipal communities shapefile***

In [22]:
# Load municipal communities shapefile
communities = gpd.read_file("C:/Users/Giorgos/Desktop/HMMY/10ο Εξάμηνο/Διπλωματική/2. Datasets/ELSTAT/ΔΗΜΟΤΙΚΕΣ ΚΟΙΝΟΤΗΤΕΣ - shapefiles/magnesia/magnesia5.shp").to_crs("EPSG:4326")
print(communities.columns)

Index(['KAL2022', 'LAU_LABEL3', 'Shape_Leng', 'Shape_Le_1', 'Shape_Area',
       'geometry'],
      dtype='object')


In [23]:
# For the municipal communities shapefile
print("Communities geometry type:", communities.geometry.geom_type.unique())

# Inspect first 5 community geometries
print("\nCommunity geometries:")
for geom in communities.geometry.head():
    if geom.geom_type == "Polygon":
        print(f"Points: {len(geom.exterior.coords)}")
    elif geom.geom_type == "MultiPolygon":
        total = sum(len(p.exterior.coords) for p in geom.geoms)
        print(f"MultiPolygon total points: {total}")

Communities geometry type: ['Polygon' 'MultiPolygon']

Community geometries:
Points: 104
Points: 97
Points: 144
Points: 185
Points: 81


## ***join grid and municipal communitites***

In [24]:
# Use centroids for spatial join
grid_centroids = grid.set_geometry('centroid_geom')
grid_with_community = gpd.sjoin(grid_centroids, communities, how='left', predicate='intersects')

# grid_with_community = gpd.sjoin(grid, communities, how='left', predicate='within')

In [25]:
grid_with_community.columns

Index(['id', 'left', 'top', 'right', 'bottom', 'row_index', 'col_index',
       'centroi_lo', 'centroi_la', 'geometry', 'centroid_geom', 'index_right',
       'KAL2022', 'LAU_LABEL3', 'Shape_Leng', 'Shape_Le_1', 'Shape_Area'],
      dtype='object')

In [27]:
grid_with_community.drop(columns=['index_right', 'KAL2022', 'Shape_Leng', 'Shape_Le_1', 'Shape_Area'], inplace=True)

In [28]:
grid_with_community.head()

,id,left,top,right,bottom,row_index,col_index,centroi_lo,centroi_la,geometry,centroid_geom,LAU_LABEL3
0,1.0,369341.6791,4.383963e+06,369841.6791,4.383463e+06,0.0,0.0,3695.0000,4383712.906,"POLYGON ((22.47999 39.59805, 22.48581 39.59813...",POINT (22.48295 39.59584),NaN
1,2.0,369341.6791,4.383463e+06,369841.6791,4.382963e+06,1.0,0.0,369591.6791,4383212.906,"POLYGON ((22.48009 39.59355, 22.48591 39.59362...",POINT (22.48305 39.59133),NaN
2,3.0,369341.6791,4.382963e+06,369841.6791,4.382463e+06,2.0,0.0,369591.6791,4382712.906,"POLYGON ((22.48019 39.58904, 22.48601 39.58912...",POINT (22.48315 39.58683),NaN
3,4.0,369341.6791,4.382463e+06,369841.6791,4.381963e+06,3.0,0.0,369591.6791,4382212.906,"POLYGON ((22.48029 39.58454, 22.48611 39.58462...",POINT (22.48325 39.58233),NaN
4,5.0,369341.6791,4.381963e+06,369841.6791,4.381463e+06,4.0,0.0,369591.6791,4381712.906,"POLYGON ((22.48038 39.58004, 22.4862 39.58011,...",POINT (22.48334 39.57782),NaN


In [29]:
grid_with_community.to_csv("grids_with_municipal_community.csv", index=False)

## check map

In [33]:
df = pd.read_csv("grids.csv")
df['geometry'] = df['geometry'].apply(wkt.loads)
df['centroid_geom'] = df['centroid_geom'].apply(wkt.loads)

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

In [34]:
print(len(gdf))
gdf.head()

21150


,id,left,top,right,bottom,row_index,col_index,centroi_lo,centroi_la,geometry,centroid_geom,Municipal_Community
0,1.0,369341.6791,4.383963e+06,369841.6791,4.383463e+06,0.0,0.0,3695.0000,4383712.906,"POLYGON ((22.47999 39.59805, 22.48581 39.59813...",POINT (22.482951074714478 39.595836715688606),NaN
1,2.0,369341.6791,4.383463e+06,369841.6791,4.382963e+06,1.0,0.0,369591.6791,4383212.906,"POLYGON ((22.48009 39.59355, 22.48591 39.59362...",POINT (22.48304932111804 39.591333108743825),NaN
2,3.0,369341.6791,4.382963e+06,369841.6791,4.382463e+06,2.0,0.0,369591.6791,4382712.906,"POLYGON ((22.48019 39.58904, 22.48601 39.58912...",POINT (22.48314754546389 39.58682949807805),NaN
3,4.0,369341.6791,4.382463e+06,369841.6791,4.381963e+06,3.0,0.0,369591.6791,4382212.906,"POLYGON ((22.48029 39.58454, 22.48611 39.58462...",POINT (22.483245747757486 39.58232588369148),NaN
4,5.0,369341.6791,4.381963e+06,369841.6791,4.381463e+06,4.0,0.0,369591.6791,4381712.906,"POLYGON ((22.48038 39.58004, 22.4862 39.58011,...",POINT (22.483343928004327 39.57782226558424),NaN


In [35]:
# Filter: keep only grids with a non-null municipal community
valid_grids = gdf[gdf['Municipal_Community'].notna()].copy()
print(len(valid_grids))
valid_grids.head()

9438


,id,left,top,right,bottom,row_index,col_index,centroi_lo,centroi_la,geometry,centroid_geom,Municipal_Community
107,108.0,369341.6791,4.330463e+06,369841.6791,4.329963e+06,107.0,0.0,369591.6791,4330212.906,"POLYGON ((22.4904 39.11614, 22.49618 39.11622,...",POINT (22.493339428258892 39.113929703050026),Δημοτική Κοινότητα Ανάβρας
111,112.0,369341.6791,4.328463e+06,369841.6791,4.327963e+06,111.0,0.0,369591.6791,4328212.906,"POLYGON ((22.49078 39.09813, 22.49657 39.0982,...",POINT (22.493722967056804 39.09591366415589),Δημοτική Κοινότητα Ανάβρας
247,248.0,369841.6791,4.330963e+06,370341.6791,4.330463e+06,106.0,1.0,370091.6791,4330712.906,"POLYGON ((22.49609 39.12072, 22.50187 39.1208,...",POINT (22.49902533839327 39.11850830370476),Δημοτική Κοινότητα Ανάβρας
248,249.0,369841.6791,4.330463e+06,370341.6791,4.329963e+06,107.0,1.0,370091.6791,4330212.906,"POLYGON ((22.49618 39.11622, 22.50196 39.11629...",POINT (22.49912090899376 39.114004291337366),Δημοτική Κοινότητα Ανάβρας
249,250.0,369841.6791,4.329963e+06,370341.6791,4.329463e+06,108.0,1.0,370091.6791,4329712.906,"POLYGON ((22.49628 39.11172, 22.50206 39.11179...",POINT (22.499216458188847 39.10950027526782),Δημοτική Κοινότητα Ανάβρας


In [36]:
# Compute mean center from centroids
centroids = valid_grids['centroid_geom']
mean_lat = centroids.apply(lambda p: p.y).mean()
mean_lon = centroids.apply(lambda p: p.x).mean()

# Create folium map
m = folium.Map(location=[mean_lat, mean_lon], zoom_start=11)

# ⚠️ Drop non-serializable columns before converting to GeoJSON
geojson_data = json.loads(valid_grids.drop(columns=['centroid_geom']).to_json())

# Add to folium
folium.GeoJson(
    geojson_data,
    tooltip=folium.GeoJsonTooltip(fields=['Municipal_Community']),
    style_function=lambda x: {
        'fillColor': 'blue',
        'color': 'black',
        'weight': 0.3,
        'fillOpacity': 0.4
    }
).add_to(m)

# Save and open in browser
m.save("valid_grids_map.html")

## ***beautify final columns***

In [47]:
df = pd.read_csv("grids.csv")

In [48]:
df.drop(columns=['centroi_lo', 'centroi_la'], inplace=True)

In [49]:
from shapely import wkt

# Parse the WKT string into actual Point objects
df['centroid_geom'] = df['centroid_geom'].apply(wkt.loads)

# Now extract lon/lat
df['centroid_lon'] = df['centroid_geom'].apply(lambda point: point.x)
df['centroid_lat'] = df['centroid_geom'].apply(lambda point: point.y)


In [51]:
df.drop(columns=['centroid_geom'], inplace=True)

In [52]:
df.head()

,id,left,top,right,bottom,row_index,col_index,geometry,Municipal_Community,centroid_lon,centroid_lat
0,1.0,369341.6791,4.383963e+06,369841.6791,4.383463e+06,0.0,0.0,POLYGON ((22.479991203666504 39.59805048281335...,NaN,22.482951,39.595837
1,2.0,369341.6791,4.383463e+06,369841.6791,4.382963e+06,1.0,0.0,POLYGON ((22.480089649443066 39.59354688377664...,NaN,22.483049,39.591333
2,3.0,369341.6791,4.382963e+06,369841.6791,4.382463e+06,2.0,0.0,POLYGON ((22.48018807311691 39.589043281018085...,NaN,22.483148,39.586829
3,4.0,369341.6791,4.382463e+06,369841.6791,4.381963e+06,3.0,0.0,POLYGON ((22.480286474693532 39.58453967453788...,NaN,22.483246,39.582326
4,5.0,369341.6791,4.381963e+06,369841.6791,4.381463e+06,4.0,0.0,POLYGON ((22.480384854178418 39.58003606433615...,NaN,22.483344,39.577822


In [53]:
df.to_csv("grids2.csv", index=False)